# Part 5 · 02 Clean and prepare

Turns the raw cache into the inputs the two readers will score. Everything is keyed by accession and covers every event; the final sample is a filter applied at the end, once the clock is final.

1. **Clean** (`clean_release.py`, versioned by `CLEAN_VERSION`): text with paragraphs, financial tables kept aside, page furniture and repeated content removed (and counted), sections labelled (safe harbor, About, contacts, conference call), `text_body` without them, sentences.
2. **Entities** (`entities.py`): names and tickers to mask, from part 3–4's WRDS cache.
3. **Tokens and pieces** (`token_stats.py`): length per model with its own tokenizer, and the pieces each text is cut into at sentence boundaries, cached in `data/chunks/<model>/`: they are the tokenized inputs the scoring step reads (`pieces()` rebuilds their text).
4. **Duplicates** (`dedup.py`): exact, near (MinHash) and close events of the same firm — flagged, never removed.
5. **Model-output cache** (`model_cache.py`): created empty, used by the scorers.
6. **Split draft** (`splits.py`): dev / validation / excluded_2020 / holdout from `common.py`, on the clock's reaction date. The `--freeze` run is not in this notebook.
7. **Manifest and checks.**

**Before running:** `01_raw_cache` has filled `cache/raw/` (the 200-event test is enough to try this notebook).

`data/entities.parquet` holds WRDS data: it stays in the team's private storage and is never part of the submission.

In [ ]:
!pip install -q beautifulsoup4 lxml nltk pyarrow datasketch tiktoken transformers
# If Colab asks you to restart the session after this install, restart it and
# continue from the next cell.

In [ ]:
import os, sys
from pathlib import Path

# Find the checkout from the repository root or any notebook subfolder.
_start = Path(os.environ.get("S08_ROOT", Path.cwd())).expanduser().resolve()
REPO_ROOT = next((p for p in (_start, *_start.parents)
                  if (p / "scripts").is_dir()
                  and (p / "3_4_surprise_returns" / "code").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Open this notebook inside the cloned S08 repository, "
                            "or set S08_ROOT to its path.")
# Optional private data root; code always comes from this checkout.
TEAM_DIR = Path(os.environ.get("TEAM_DIR", REPO_ROOT)).expanduser().resolve()
os.environ["TEAM_DIR"] = str(TEAM_DIR)
SCRIPTS = REPO_ROOT / "scripts"
NOV2_DATA = TEAM_DIR / "1_2nov" / "nov2_data"
if not NOV2_DATA.is_dir():
    NOV2_DATA = TEAM_DIR / "files csv"

STEP5_DIR = TEAM_DIR / "5_data cleaning"
CODE = REPO_ROOT / "5_data cleaning" / "code"
STEP5_DIR.mkdir(parents=True, exist_ok=True)

for sub in ["cache", "data", "results", "logs"]:
    (STEP5_DIR / sub).mkdir(parents=True, exist_ok=True)

os.environ["TEAM_DIR"] = str(TEAM_DIR)
os.environ["STEP5_DIR"] = str(STEP5_DIR)
sys.path.insert(0, str(CODE))
os.chdir(STEP5_DIR)       # relative artifacts stay in the private data folder

missing = [f for f in ["settings.py", "cache_raw.py", "clean_release.py", "test_clean_release.py",
                                 "entities.py", "token_stats.py", "dedup.py", "model_cache.py",
                                 "splits.py", "manifest.py"] if not (CODE / f).exists()]
assert not missing, f"Missing code in {CODE}: {missing}"
print("step 5 folder:", STEP5_DIR)
print("code files:   ", sorted(os.listdir(CODE)))

## 1. Clean

Years already cleaned with the current `CLEAN_VERSION` are skipped; add `--force` after changing the rules (and bump `CLEAN_VERSION`).

In [ ]:
__import__("subprocess").run([sys.executable, str(CODE / 'clean_release.py')], check=True)

## 2. Entities to mask

In [ ]:
__import__("subprocess").run([sys.executable, str(CODE / 'entities.py')], check=True)

## 3. Token counts and pieces

Writes the pieces of every text per model to `data/chunks/<model>/chunks_<year>.parquet`; a year whose pieces already match the cleaned texts and the window settings is skipped. FinBERT and ChronoGPT need no login. Llama 3.1 is gated: add a Hugging Face token with access to `meta-llama/Llama-3.1-8B-Instruct` as the Colab secret `HF_TOKEN` (key icon on the left); without it, Llama is skipped.

In [ ]:
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("no HF_TOKEN secret: Llama 3.1 will be skipped")

__import__("subprocess").run([sys.executable, str(CODE / 'token_stats.py')], check=True)

## 4. Duplicates and close events

In [ ]:
__import__("subprocess").run([sys.executable, str(CODE / 'dedup.py')], check=True)

## 5. Model-output cache and 6. split draft

In [ ]:
__import__("subprocess").run([sys.executable, str(CODE / 'model_cache.py')], check=True)
__import__("subprocess").run([sys.executable, str(CODE / 'splits.py')], check=True)

## 7. Manifest and checks

Each check recomputes a figure from the files and compares it with what it should be. A step not run yet is reported as not produced and skipped.

In [ ]:
__import__("subprocess").run([sys.executable, str(CODE / 'manifest.py')], check=True)

In [ ]:
import glob, hashlib, json, zipfile
import pandas as pd
import pyarrow.parquet as pq
import settings as S
import splits

ok, bad = 0, []


def check(label, got, want, pct=None):
    global ok
    hit = (round(got, 4) == round(want, 4))
    if hit:
        ok += 1
    else:
        bad.append(f"{label}: got {got}, expected {want}")
    tail = f"  ({pct:.1f}%)" if pct is not None else ""
    print(f"  {'ok ' if hit else 'NO '}{label:<52}{got:>12,}{tail}")


def have(p):
    if p.exists():
        return True
    print(f"  --  {p.name} not produced yet, skipped")
    return False


# The event universe is the release clock: one row per accession.
clk = pd.read_csv(S.CLOCK, usecols=["accession", "reaction_date"], dtype=str)
n_events = int(clk.accession.nunique())
check("events in the release clock, one row each", len(clk), n_events)

old = pd.read_csv(S.EXTRACTION_REPORT)
old_src = old.source.fillna("")
old_got = old_src.ne("none") & old_src.ne("index fetch failed")

print("\nraw cache")
n_stored = None
if have(S.RAW_MANIFEST):
    docs = pd.read_parquet(S.RAW_MANIFEST).query("kind == 'doc'")
    stored = docs.sha256.fillna("").ne("")
    members = set()
    for z in sorted(glob.glob(str(S.RAW_DIR / "raw_*.zip"))):
        with zipfile.ZipFile(z) as zz:
            members.update(m for m in zz.namelist() if not m.endswith("__index.htm"))
    n_stored = int(stored.sum())
    print(f"  documents cached: {n_stored:,} ({100 * n_stored / n_events:.1f}% of the events); "
          f"events not fetched yet: {n_events - len(docs):,}")
    check("archive members, against the manifest", len(members), n_stored)
    check("manifest rows per accession, duplicates", len(docs) - int(docs.accession.nunique()), 0)

print("\nclean texts")
cl = None
if have(S.CLEAN_DIR):
    cols = ["accession", "clean_version", "error", "n_words", "n_words_body",
            "n_words_tables", "n_sentences", "n_replacement_chars", "sha256_text",
            "n_furniture_removed", "n_repeats_removed", "n_table_copies_removed"]
    cl = pd.concat([pd.read_parquet(f, columns=cols)
                    for f in sorted(glob.glob(str(S.CLEAN_DIR / "clean_*.parquet")))],
                   ignore_index=True)
    if n_stored is not None:
        check("texts cleaned, against documents cached", len(cl), n_stored)
    check("cleaning errors", int(cl.error.fillna("").ne("").sum()), 0)
    check("clean versions present", int(cl.clean_version.nunique()), 1)
    check("accessions cleaned twice", len(cl) - int(cl.accession.nunique()), 0)
    check("texts with replacement characters", int(cl.n_replacement_chars.fillna(0).gt(0).sum()), 0)
    check("texts with an empty body", int(cl.n_words_body.fillna(0).eq(0).sum()), 0)
    for col, what in (("n_furniture_removed", "page furniture lines"),
                      ("n_repeats_removed", "repeated paragraphs"),
                      ("n_table_copies_removed", "tables printed twice")):
        n = cl[col].fillna(0)
        print(f"  removed {what:<22} {int(n.sum()):>9,} in {int(n.gt(0).sum()):,} texts")
    # The old text kept statement tables with long labels in the prose; the new one
    # sets them aside, so it is compared on text plus tables. The shared report has no
    # accession, so this is by distribution; the full report allows it event by event.
    cl["n_words_all"] = cl.n_words + cl.n_words_tables
    for name, s in (("words, fetch_releases", old.loc[old_got, "words"]),
                    ("words, text + tables", cl.n_words_all), ("words, text_full", cl.n_words),
                    ("words, text_body", cl.n_words_body), ("sentences", cl.n_sentences)):
        q = s.quantile([0.10, 0.50, 0.90])
        print(f"  {name:<22} p10 {int(q[0.10]):>7,}  median {int(q[0.50]):>7,}  p90 {int(q[0.90]):>7,}")
    full = S.extraction_full()
    if full:
        m = cl.merge(pd.read_csv(full, usecols=["accession", "words"]), on="accession")
        m = m[m.words > 0]
        short = m.n_words_all < 0.80 * m.words      # content lost, not moved to tables
        check("texts with under 80% of the old words (lost content)", int(short.sum()), 0)
        out = S.RESULTS / "clean_shorter_than_old.csv"
        m.loc[short, ["accession", "words", "n_words", "n_words_tables"]].to_csv(out, index=False)
        print(f"  correlation with the old word counts: {m.n_words_all.corr(m.words):.3f}; "
              f"short texts listed in {out.name}")

print("\ntokens")
if have(S.TOKEN_STATS) and have(S.TOKEN_QUANTILES):
    ts, tq = pd.read_parquet(S.TOKEN_STATS), pd.read_csv(S.TOKEN_QUANTILES)
    if cl is not None:
        check("texts with token counts", len(ts), int(cl.error.fillna("").eq("").sum()))
    for model in tq.model.unique():
        med = ts.groupby("year")[f"tokens_{model}"].median().astype(int)
        q = tq[tq.model == model].set_index("year").tokens_p50
        check(f"{model}: yearly medians matching the per-text file",
              int((med.reindex(q.index) == q).sum()), len(q))
        print(f"  {model:<11} median {int(ts[f'tokens_{model}'].median()):>7,} tokens, "
              f"one piece {100 * ts[f'chunks_{model}'].eq(1).mean():.1f}%")

print("\npieces (tokenized inputs)")
if have(S.CHUNK_DIR):
    for mdir in sorted(p for p in S.CHUNK_DIR.iterdir() if p.is_dir()):
        files = sorted(mdir.glob("chunks_*.parquet"))
        if not files:
            continue
        ch = pd.concat([pd.read_parquet(f, columns=["accession", "sha256_text", "clean_version",
                                                    "max_tokens", "chunk_idx", "n_tokens",
                                                    "truncated"]) for f in files],
                       ignore_index=True)
        m = mdir.name
        if cl is not None:
            ok_texts = cl[cl.error.fillna("").eq("")]
            check(f"{m}: texts with pieces, against texts cleaned",
                  int(ch.accession.nunique()), len(ok_texts))
            cur = set(zip(ok_texts.accession, ok_texts.sha256_text))
            per_text = ch.drop_duplicates("accession")
            check(f"{m}: texts whose pieces were cut from an older text",
                  int((~pd.Series(list(zip(per_text.accession, per_text.sha256_text)))
                       .isin(cur)).sum()), 0)
        over = (ch.n_tokens > ch.max_tokens) & ~ch.truncated
        check(f"{m}: pieces over the window, not flagged", int(over.sum()), 0)
        print(f"  {m:<11} {int((ch.chunk_idx >= 0).sum()):>9,} pieces, "
              f"{int(ch.truncated.sum()):,} truncated sentences, window {int(ch.max_tokens.iloc[0]):,}")

print("\nduplicates")
if have(S.DEDUP_FLAGS):
    dd = pd.read_parquet(S.DEDUP_FLAGS)
    if cl is not None:
        exact = int(cl.groupby("sha256_text").accession.transform("size").gt(1).sum())
        check("texts in an exact-duplicate group", int(dd.flag_type.eq("exact").sum()), exact)
    for kind in ("exact", "near", "close_events"):
        g = dd[dd.flag_type == kind]
        print(f"  {kind:<13}{len(g):>8,} accessions in {g.dup_group.nunique():,} groups")

print("\nsplit draft")
if have(S.SPLITS_DRAFT):
    common = S.common_3_4()
    sp = pd.read_parquet(S.SPLITS_DRAFT)
    rule = splits.label(sp.date_used, common)
    check("events whose split breaks common.py's rule", int((rule != sp.split).sum()), 0)
    check("events in the split draft", len(sp), n_events)
    for s in list(common.SPLITS) + ["no_date"]:
        check(f"split {s}", int(sp.split.eq(s).sum()), int(rule.eq(s).sum()),
              100 * sp.split.eq(s).mean())
    print(sp.date_source.value_counts().to_string())
    if S.HOLDOUT_IDS.exists():
        want = S.HOLDOUT_IDS.with_suffix(".sha256").read_text().split()[0]
        check("frozen holdout list matches its sha256",
              int(hashlib.sha256(S.HOLDOUT_IDS.read_bytes()).hexdigest() == want), 1)

print("\nmanifest")
if have(S.MANIFEST):
    mf = json.loads(S.MANIFEST.read_text())
    rows = {S.STEP5_DIR / f["path"]: f["rows"] for f in mf["files"] if f["path"].endswith(".parquet")}
    same = sum(p.exists() and pq.ParquetFile(p).metadata.num_rows == n for p, n in rows.items())
    check("parquet files whose rows match manifest.json", same, len(rows))

print(f"\n{ok} checks agree" + (f", {len(bad)} do not:" if bad else ""))
for b in bad:
    print("  ", b)